# 🔧 02 — Prepare Scenes
### Gaza Building Damage Assessment Pipeline

Loads the Maxar VHR imagery, chips it into DOFA-compatible patches,
and aligns pre/post scenes to the same spatial grid.

---
**Pipeline:**
1. ✅ `01_download_data.ipynb`
2. 🔧 `02_prepare_scenes.ipynb` ← *you are here*
3. 🏚️ `03_assess_damage.ipynb`
4. 🎨 `04_visualise.ipynb`


## ⚙️ Configuration

In [ ]:
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.transform import from_bounds
from pathlib import Path
from PIL import Image
import warnings
warnings.filterwarnings("ignore")

RAW_DIR      = Path("data/raw")
PREPARED_DIR = Path("data/prepared")
CHIPS_DIR    = Path("data/chips")
PREPARED_DIR.mkdir(parents=True, exist_ok=True)
CHIPS_DIR.mkdir(parents=True, exist_ok=True)

# DOFA patch size — must match training config
PATCH_SIZE = 64

# Overlap between chips (helps at building boundaries)
STRIDE = 48   # 75% overlap

print("✅ Configuration set")
for period in ["pre", "post"]:
    p = RAW_DIR / f"gaza_{period}_event.tif"
    print(f"   {period}-event: {'✅ found' if p.exists() else '❌ missing — run notebook 01'}")


## 🔍 Step 1 — Inspect Scenes

In [ ]:
import matplotlib.pyplot as plt

def load_scene(period):
    path = RAW_DIR / f"gaza_{period}_event.tif"
    with rasterio.open(path) as src:
        data    = src.read()            # (3, H, W)
        profile = src.profile
        bounds  = src.bounds
        res     = src.res
    return data, profile, bounds, res

pre_data,  pre_profile,  pre_bounds,  pre_res  = load_scene("pre")
post_data, post_profile, post_bounds, post_res = load_scene("post")

print("Pre-event scene:")
print(f"   Shape     : {pre_data.shape}")
print(f"   Resolution: {pre_res[0]:.6f}° (~{pre_res[0]*111000:.1f}m per pixel)")
print(f"   Bounds    : {pre_bounds}")
print(f"   Dtype     : {pre_data.dtype}")

print("\nPost-event scene:")
print(f"   Shape     : {post_data.shape}")
print(f"   Resolution: {post_res[0]:.6f}° (~{post_res[0]*111000:.1f}m per pixel)")
print(f"   Bounds    : {post_bounds}")


## 📐 Step 2 — Normalise to 0-1 Float

In [ ]:
def normalise(data):
    """Normalise uint8 or uint16 to float32 0-1."""
    data = data.astype(np.float32)
    if data.max() > 1.0:
        data = data / (255.0 if data.max() <= 255 else 65535.0)
    return np.clip(data, 0, 1)

pre_norm  = normalise(pre_data)
post_norm = normalise(post_data)

# Crop to minimum common size so both arrays match
min_h = min(pre_norm.shape[1], post_norm.shape[1])
min_w = min(pre_norm.shape[2], post_norm.shape[2])
pre_norm  = pre_norm[:,  :min_h, :min_w]
post_norm = post_norm[:, :min_h, :min_w]

print(f"✅ Normalised — shape: {pre_norm.shape}")
print(f"   Pre  range: [{pre_norm.min():.3f}, {pre_norm.max():.3f}]")
print(f"   Post range: [{post_norm.min():.3f}, {post_norm.max():.3f}]")

# Save aligned normalised scenes
def save_tif(data, path, profile):
    profile = profile.copy()
    profile.update(count=data.shape[0], dtype="float32",
                   height=data.shape[1], width=data.shape[2],
                   driver="GTiff", compress="lzw")
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(data.astype(np.float32))
    print(f"   Saved: {path}")

save_tif(pre_norm,  PREPARED_DIR / "pre_rgb.tif",  pre_profile)
save_tif(post_norm, PREPARED_DIR / "post_rgb.tif", post_profile)


## 🔲 Step 3 — Chip into DOFA Patches

In [ ]:
import json

def chip_scene(data, period, patch_size=PATCH_SIZE, stride=STRIDE):
    """Slide a window across the scene and save each chip."""
    _, H, W  = data.shape
    chip_dir = CHIPS_DIR / period
    chip_dir.mkdir(exist_ok=True)

    chips_meta = []
    chip_id    = 0

    for y in range(0, H - patch_size + 1, stride):
        for x in range(0, W - patch_size + 1, stride):
            chip = data[:, y:y+patch_size, x:x+patch_size]   # (3, 64, 64)

            chip_path = chip_dir / f"chip_{chip_id:05d}.npy"
            np.save(chip_path, chip.astype(np.float32))

            chips_meta.append({
                "chip_id": chip_id,
                "period":  period,
                "y":       y, "x": x,
                "h":       patch_size, "w": patch_size,
                "path":    str(chip_path),
            })
            chip_id += 1

    print(f"   {period}: {chip_id} chips saved to {chip_dir}")
    return chips_meta

print("✂️  Chipping scenes into 64×64 patches...")
pre_meta  = chip_scene(pre_norm,  "pre")
post_meta = chip_scene(post_norm, "post")

# Save metadata
meta = {"pre": pre_meta, "post": post_meta,
        "patch_size": PATCH_SIZE, "stride": STRIDE,
        "scene_shape": list(pre_norm.shape)}
with open(PREPARED_DIR / "chips_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"\n✅ Total chips: {len(pre_meta)} pre + {len(post_meta)} post")
print(f"   Metadata saved to {PREPARED_DIR}/chips_meta.json")


## 🖼️ Step 4 — Preview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.suptitle("Gaza — Maxar VHR Imagery: Pre vs Post Event", fontsize=14, fontweight="bold")

for ax, data, period, color in zip(axes, [pre_norm, post_norm], ["Pre-event (Oct 2023)", "Post-event (Jan 2024)"], ["#2ecc71", "#e74c3c"]):
    img = np.transpose(data, (1, 2, 0)).clip(0, 1)
    ax.imshow(img)
    ax.set_title(period, fontsize=13, fontweight="bold", color=color)
    ax.axis("off")
    for s in ax.spines.values():
        s.set_edgecolor(color); s.set_linewidth(3); s.set_visible(True)

plt.tight_layout()
plt.savefig(PREPARED_DIR / "preview.png", dpi=120, bbox_inches="tight")
plt.show()

# Show sample chips
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
fig.suptitle("Sample 64×64 Chips — Pre (top) vs Post (bottom)", fontsize=12, fontweight="bold")
sample_ids = np.linspace(0, len(pre_meta)-1, 6, dtype=int)

for col, idx in enumerate(sample_ids):
    pre_chip  = np.load(pre_meta[idx]["path"]).transpose(1,2,0).clip(0,1)
    post_chip = np.load(post_meta[idx]["path"]).transpose(1,2,0).clip(0,1)
    axes[0, col].imshow(pre_chip);  axes[0, col].axis("off")
    axes[1, col].imshow(post_chip); axes[1, col].axis("off")

axes[0, 0].set_ylabel("Pre", fontsize=11, fontweight="bold")
axes[1, 0].set_ylabel("Post", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()
print("\n✅ Proceed to: 03_assess_damage.ipynb")
